In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from tqdm import tqdm
from sklearn.linear_model import LinearRegression,ElasticNet
from sklearn.ensemble import VotingRegressor,BaggingRegressor,RandomForestRegressor
import os
os.chdir("/home/pgcp-ai/MachineLearning/Datasets/RoadAccident/")

In [4]:
accident = pd.read_csv("train.csv", index_col = 0)
accident

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
id,,,,,,,,,,,,,
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...
517749,highway,4,0.10,70,daylight,foggy,True,True,afternoon,False,False,2,0.32
517750,rural,4,0.47,35,daylight,rainy,True,True,morning,False,False,1,0.26
517751,urban,4,0.62,25,daylight,foggy,False,False,afternoon,False,True,0,0.19


In [5]:
accident.isna().sum().sum()

0

In [6]:
X, y = accident.drop('accident_risk', axis = 1), accident['accident_risk']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26)

In [8]:
ohe = OneHotEncoder(sparse_output = False, drop = 'first',handle_unknown='ignore').set_output(transform = 'pandas')
transformer = ColumnTransformer(transformers=[('OHE',ohe,make_column_selector(dtype_include=object)),
                                            ]
                               ,remainder='passthrough',
                               verbose_feature_names_out=False
                               ).set_output(transform='pandas')

In [9]:
X_train = transformer.fit_transform(X_train)
X_test = transformer.transform(X_test)

In [10]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
y_pred[y_pred < 0] = 0
y_pred.min()

0.0

In [11]:
ss = StandardScaler()

In [12]:
X = transformer.fit_transform(X)
X = ss.fit_transform(X)
bm = LinearRegression()
bm.fit(X, y)

LinearRegression()

In [13]:
tst = pd.read_csv("test.csv", index_col = 0)
tst_ohe = transformer.transform(tst)
tst_ss = ss.transform(tst_ohe)
y_pred = bm.predict(tst_ss)

In [14]:
submit = pd.read_csv("sample_submission.csv")
submit["accident_risk"] = y_pred

In [15]:
submit.to_csv("KaggleSubmission.csv", index = False)

In [16]:
ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

In [17]:
lr = LinearRegression()
el = ElasticNet(alpha=0.05,l1_ratio=0.001)
dtr = DecisionTreeRegressor(max_depth=5,random_state=26)

In [18]:
estimators = [dtr,lr,el]
for e in estimators:
    e.fit(X_train, y_train)
    y_pred = e.predict(X_test)
    print("R2 Score of ", e, "=", r2_score(y_test, y_pred))

R2 Score of  DecisionTreeRegressor(max_depth=5, random_state=26) = 0.8169967616753644
R2 Score of  LinearRegression() = 0.8042445527598957
R2 Score of  ElasticNet(alpha=0.05, l1_ratio=0.001) = 0.8013980340987192


In [19]:
lr = LinearRegression()
el = ElasticNet(alpha=0.05,l1_ratio=0.001)
dtr = DecisionTreeRegressor(max_depth=5,random_state=26)

voting = VotingRegressor(estimators = [
     (
        "TREE",
        dtr
    ),
     (
        "LR",
        lr
    ),
    (
        "EL",
        el
    )
],weights = [0.9,0.6,0.8])
voting.fit(X_train,y_train)

VotingRegressor(estimators=[('TREE',
                             DecisionTreeRegressor(max_depth=5,
                                                   random_state=26)),
                            ('LR', LinearRegression()),
                            ('EL', ElasticNet(alpha=0.05, l1_ratio=0.001))],
                weights=[0.9, 0.6, 0.8])

In [20]:
y_pred = voting.predict(X_test)
r2_score(y_test,y_pred)

0.8429124235921474

In [21]:
bagg = BaggingRegressor(estimator=dtr,n_estimators=100,n_jobs=-1)
bagg.fit(X_train,y_train)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=5, random_state=26),
                 n_estimators=100, n_jobs=-1)

In [22]:
y_pred = bagg.predict(X_test)
r2_score(y_test,y_pred)

0.8190775810882864

In [28]:
rf = RandomForestRegressor(n_estimators=100,n_jobs=-1)
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
r2_score(y_test,y_pred)

0.8718228470670217

In [29]:
np.sqrt(mean_squared_error(y_test,y_pred))

0.059534581547407865

In [32]:
rf.fit(X, y)

RandomForestRegressor(n_jobs=-1)

In [33]:
y_pred = rf.predict(tst_ss)

In [35]:
submit["accident_risk"] = y_pred
submit

,id,accident_risk
0,517754,0.337217
1,517755,0.132700
2,517756,0.161650
3,517757,0.322700
4,517758,0.404800
...,...,...
172580,690334,0.102100
172581,690335,0.445800
172582,690336,0.209317
172583,690337,0.123200


In [36]:
submit.to_csv("KaggleSubmissionWithRandomForest.csv", index = False)